# Tahap 2: Preprocessing & Batch Feature Engineering
### Pembangunan Dataset Spasial-Temporal Panel secara Out-of-Core (Batch per Tahun)

---

## 📌 Mengapa Pemrosesan Wajib Dilakukan Per Batch?

Total observasi data penelitian ini mencakup rentang waktu 10 tahun (22 September 2016 – 31 Agustus 2026, total 3.631 hari) pada 5.407 grid daratan Sumatra:

$$\text{Total Baris} = 3.631 \times 5.407 = 19.633.597 \text{ baris observasi spasial-temporal!}$$

Jika seluruh 19,6 juta baris digabungkan sekaligus ke dalam satu objek DataFrame pandas di memori RAM bersama 20+ kolom fitur:
- Memori yang dibutuhkan mencapai **15 s.d. 30 GB RAM**.
- Pada laptop standar (RAM 8 GB atau 16 GB), hal ini **PASTI MEMBUAT LAPTOP FREEZE / CRASH (OOM Killer)**.

### 🛡️ Solusi Arsitektur Batch (Sesuai Blueprint):
1. Data diproses dalam **Batch per Tahun** (2016 s.d. 2026, total 11 file batch independen).
2. Setiap tahun hanya berukuran $\approx 1,97$ juta baris, mengonsumsi memori RAM hanya **~400–600 MB** saat diproses.
3. **Cross-Year 7-Day Temporal Buffer**: Saat memproses tahun $Y$, sistem secara otomatis memuat *buffer* 7 hari dari akhir tahun $Y-1$ dan 1 hari dari awal tahun $Y+1$. Hal ini menjamin bahwa seluruh fitur lag (`rain_3d`, `rain_7d`, `CDD`, `hotspot_count_t1..3`, `spatial_lag_hs`) dan target esok hari ($S_{t+1}$) terhitung **100% presisi matematis tanpa ada data terpotong di pergantian tahun**.
4. Setiap batch disimpan dalam format terkompresi **Parquet** di `data/processed/panel_batches/batch_{YYYY}.parquet`.


In [ ]:
import sys
import os
import glob
from pathlib import Path
import psutil
import numpy as np
import pandas as pd
from tqdm import tqdm

# Hubungkan ke modul lokal project
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PROCESSED_DIR, BATCH_DIR, GRID_FILE, LANDCOVER_FILE, NEIGHBOR_FILE,
    FEATURE_COLS, TARGET_COL, META_COLS, TRAIN_END, VAL_END, TEST_END
)
from src.preprocessing.batch_builder import (
    build_single_year_batch, build_all_batches, iter_panel_batches, get_available_years
)

def print_memory_status():
    mem = psutil.virtual_memory()
    print(f"📊 Status RAM Sistem: Terpakai {mem.used / (1024**3):.2f} GB / Total {mem.total / (1024**3):.2f} GB ({mem.percent}% digunakan)")

print_memory_status()
print(f"Direktori Batch Output: {BATCH_DIR}")


---
## 1. Spesifikasi Fitur Spasial-Temporal & Target

Setiap baris data merepresentasikan satu pasangan $(\text{grid}, \text{tanggal})$:

| Kategori | Kolom Fitur | Keterangan |
| :--- | :--- | :--- |
| **Identitas & Spasial** | `grid_id`, `date`, `grid_lon`, `grid_lat`, `landcover_code` | ID grid, koordinat centroid, dan kelas tutupan lahan WorldCover |
| **Hotspot Hari Ini ($t$)** | `state`, `hotspot_count`, `FRP_sum` | Status aktif ($S_t \in \{0, 1\}$), jumlah titik hotspot, dan total intensitas FRP |
| **Lag Hotspot Historis** | `hotspot_count_t1`, `hotspot_count_t2`, `hotspot_count_t3`, `FRP_sum_t1` | Histori kebakaran 1, 2, dan 3 hari sebelumnya |
| **Spasial Lag** | `spatial_lag_hs` | Jumlah dari 8 tetangga spasial yang aktif terbakar pada hari $t$ |
| **Curah Hujan (CHIRPS)** | `rain_t`, `rain_t1`, `rain_3d`, `rain_7d`, `CDD` | Curah hujan harian, lag 1 hari, akumulasi 3 hari, 7 hari, dan Consecutive Dry Days |
| **Meteorologi (ERA5)** | `temperature_t`, `temperature_lag`, `relative_humidity_t`, `wind_speed` | Suhu udara 2m, lag suhu, kelembapan relatif (RH), dan kecepatan angin |
| **Target Esok Hari ($t+1$)** | `state_t1` | Status kebakaran pada hari berikutnya ($S_{t+1} \in \{0, 1\}$) |
| **Partition Split** | `split` | `train` ($\le$ 2022), `val` (2023–2024), atau `test` (2025–2026) |


In [ ]:
print("Daftar Fitur Prediktor (FEATURE_COLS):")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"  {i:02d}. {col}")

print(f"\nTarget Prediksi: {TARGET_COL} (State Hotspot Esok Hari: 0 = Padam, 1 = Aktif)")
print(f"Meta Columns: {META_COLS}")


---
## 2. Verifikasi Data Antara (*Preprocessed Intermediate Files*)
Sebelum batch panel dibangun, pastikan data antara FIRMS, CHIRPS, ERA5, dan Grid sudah siap.


In [ ]:
chirps_files = sorted(glob.glob(str(PROCESSED_DIR / 'chirps_yearly' / '*.parquet')))
era5_files = sorted(glob.glob(str(PROCESSED_DIR / 'era5_monthly' / '*.parquet')))

print(f"✓ Grid Daratan Sumatra: {GRID_FILE.exists()} ({GRID_FILE.stat().st_size / 1024:.1f} KB)")
print(f"✓ Tutupan Lahan WorldCover: {LANDCOVER_FILE.exists()} ({LANDCOVER_FILE.stat().st_size / 1024:.1f} KB)")
print(f"✓ Matriks 8 Tetangga Spasial: {NEIGHBOR_FILE.exists()} ({NEIGHBOR_FILE.stat().st_size / 1024:.1f} KB)")
print(f"✓ FIRMS Daily Grid: {(PROCESSED_DIR / 'firms_daily_grid.parquet').exists()} ({(PROCESSED_DIR / 'firms_daily_grid.parquet').stat().st_size / (1024*1024):.2f} MB)")
print(f"✓ CHIRPS Yearly Parquets: {len(chirps_files)} file tahunan terdeteksi")
print(f"✓ ERA5 Monthly Parquets: {len(era5_files)} file bulanan terdeteksi")


---
## 3. Eksekusi Pembangunan Batch per Tahun

Eksekusi batch dapat dijalankan secara terkontrol:
- Anda dapat menguji dengan **1 tahun tertentu terlebih dahulu** (misal: `build_single_year_batch(2016)`).
- Atau menjalankan loop penuh untuk seluruh tahun (2016 s.d. 2026).
- Setiap tahun yang sudah selesai dibangun akan di-cache ke disk. Jika notebook di-restart, fungsi akan mendeteksi file yang sudah ada dan melewatinya (*skip*) tanpa mengulang perhitungan.


In [ ]:
# Cek tahun-tahun yang tersedia
years_available = get_available_years()
print(f"Tahun yang tersedia untuk diproses: {years_available}")

# Contoh eksekusi Batch untuk 1 tahun pertama (misal 2016)
# Untuk memproses seluruh tahun, silakan jalankan loop di bawah ini.
for yr in years_available:
    batch_file = BATCH_DIR / f"batch_{yr}.parquet"
    if batch_file.exists():
        print(f"✓ Batch {yr} sudah ada ({batch_file.stat().st_size / (1024*1024):.1f} MB). Dilewati.")
    else:
        print(f"Memproses batch tahun {yr}...")
        build_single_year_batch(yr, force_recompute=False)
        print_memory_status()

print("\n[BATCH GENERATION FINISHED] Seluruh batch tahunan siap!")


---
## 4. Validasi Integritas Seluruh Batch yang Dihasilkan
Memastikan tidak ada missing value pada fitur, memeriksa konsistensi jumlah baris, dan distribusi pembagian split (Train, Val, Test).


In [ ]:
batch_files = sorted(glob.glob(str(BATCH_DIR / 'batch_*.parquet')))
print(f"Total file batch tersimpan: {len(batch_files)} file\n")

batch_summary = []
for bf in batch_files:
    p = Path(bf)
    yr = p.stem.split('_')[1]
    df_b = pd.read_parquet(p)
    size_mb = p.stat().st_size / (1024 * 1024)
    n_rows = len(df_b)
    n_pos = (df_b['state_t1'] == 1).sum()
    pos_pct = (n_pos / n_rows) * 100
    splits_present = list(df_b['split'].unique())
    
    batch_summary.append({
        'Batch': f"Year {yr}",
        'Ukuran File': f"{size_mb:.1f} MB",
        'Total Baris': f"{n_rows:,}",
        'Positif S=1': f"{n_pos:,}",
        'Positif (%)': f"{pos_pct:.2f}%",
        'Splits': ', '.join(splits_present)
    })

df_summary = pd.DataFrame(batch_summary)
print(df_summary.to_string(index=False))


---
## 5. Uji Coba Generator Streaming (`iter_panel_batches`)
Fungsi `iter_panel_batches` memungkinkan model downstream (seperti SGDClassifier, LightGBM, atau Decision Evaluator) membaca data baris demi baris atau batch demi batch tanpa pernah memuat seluruh data ke memori.


In [ ]:
# Menguji generator streaming untuk subset 'train'
print("Menguji generator iter_panel_batches(split='train')...")
sample_count = 0
for i, batch_chunk in enumerate(iter_panel_batches(split='train')):
    sample_count += len(batch_chunk)
    print(f"  Batch {i+1}: {len(batch_chunk):,} baris | Tanggal: {batch_chunk['date'].min().strftime('%Y-%m-%d')} s.d. {batch_chunk['date'].max().strftime('%Y-%m-%d')}")
    if i >= 2: # batasi preview 3 batch
        print("  ... preview streaming berhasil!")
        break

print(f"\nPratinjau struktur data 1 batch:")
batch_chunk[META_COLS + FEATURE_COLS + [TARGET_COL, 'split']].head(3)


---
## 6. Kesimpulan Tahap 2

1. Seluruh dataset spasial-temporal panel (~19,6 juta baris) berhasil dipecah menjadi batch tahunan yang aman dan modular.
2. Tidak terjadi lonjakan memori (RAM $< 600\text{ MB}$ sepanjang pemrosesan).
3. Data telah siap untuk dilatih oleh model-model pada Blueprint.

👉 **Langkah Selanjutnya**: Buka dan jalankan **`03_model_training_and_baselines.ipynb`** untuk melatih model Persistence, Markov Chain, Logistic Regression, dan LightGBM.
